# Build Network Term Definition Corpus

This notebook creates the term corpus used by network-term ranking. It extracts deduplicated terms from `network_name`, `cognitive_terms`, and `region_terms` in `network_test_set_labels.csv`, then fills definitions from CogAtlas and MeSH first. Any terms still missing definitions can be filled with a local Ollama model, following the pattern in `experiments/llm_relation_extraction.ipynb`.

Output:

`experiments/evaluation_resources/networks_labels/network_terms_with_definitions.csv`


In [1]:
from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

import sys
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

RESOURCE_DIR = REPO_ROOT / 'experiments' / 'evaluation_resources' / 'networks_labels'
NETWORK_LABELS_CSV = RESOURCE_DIR / 'network_test_set_labels.csv'
OUTPUT_CSV = RESOURCE_DIR / 'network_terms_with_definitions.csv'
OUTPUT_JSON = RESOURCE_DIR / 'network_terms_with_definitions.json'
MISSING_CSV = RESOURCE_DIR / 'network_terms_missing_definitions.csv'

RESOURCE_DIR.mkdir(parents=True, exist_ok=True)
print('Network labels:', NETWORK_LABELS_CSV)
print('Output CSV:', OUTPUT_CSV)


Network labels: /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_test_set_labels.csv
Output CSV: /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_terms_with_definitions.csv


In [2]:
from neurovlm.semantic_evaluation import (
    NETWORK_TERM_COLUMNS,
    build_network_term_corpus_from_label_table,
    normalize_network_term,
    split_network_term_cell,
)

labels_df = pd.read_csv(NETWORK_LABELS_CSV)
labels_df.head()


,raw_network_label,network_key,network_name,mapped_terms,region_terms,cognitive_terms,short_definition,long_definition
0,DorsAttn,attention,Attention,Attention; attention; dorsal attention; dattn;...,Intraparietal sulcus; Superior parietal lobule...,Selective attention; Spatial orienting; Sustai...,Dorsal attention network for selective attenti...,The attention network emphasizes dorsal attent...
1,DorsAttnA,attention,Attention,Attention; attention; dorsal attention; dattn;...,Intraparietal sulcus; Superior parietal lobule...,Selective attention; Spatial orienting; Sustai...,Dorsal attention network for selective attenti...,The attention network emphasizes dorsal attent...
2,DorsAttnB,attention,Attention,Attention; attention; dorsal attention; dattn;...,Intraparietal sulcus; Superior parietal lobule...,Selective attention; Spatial orienting; Sustai...,Dorsal attention network for selective attenti...,The attention network emphasizes dorsal attent...
3,dATN-A,attention,Attention,Attention; attention; dorsal attention; dattn;...,Intraparietal sulcus; Superior parietal lobule...,Selective attention; Spatial orienting; Sustai...,Dorsal attention network for selective attenti...,The attention network emphasizes dorsal attent...
4,dATN-B,attention,Attention,Attention; attention; dorsal attention; dattn;...,Intraparietal sulcus; Superior parietal lobule...,Selective attention; Spatial orienting; Sustai...,Dorsal attention network for selective attenti...,The attention network emphasizes dorsal attent...


In [3]:
# Extract deduplicated candidate terms from network_name, cognitive_terms, and region_terms.
term_rows = []
for _, row in labels_df.iterrows():
    for column in NETWORK_TERM_COLUMNS:
        for term in split_network_term_cell(row.get(column)):
            norm = normalize_network_term(term)
            if not norm:
                continue
            term_rows.append({
                'term': term,
                'normalized_term': norm,
                'source_column': column,
                'raw_network_label': row.get('raw_network_label'),
                'network_key': row.get('network_key'),
                'network_name': row.get('network_name'),
            })

term_mentions = pd.DataFrame(term_rows)
term_summary = (
    term_mentions
    .groupby('normalized_term', as_index=False)
    .agg(
        term=('term', 'first'),
        source_columns=('source_column', lambda x: '; '.join(sorted(set(map(str, x))))),
        network_keys=('network_key', lambda x: '; '.join(sorted(set(map(str, x))))),
        n_mentions=('term', 'size'),
    )
    .sort_values('term', key=lambda s: s.str.lower())
    .reset_index(drop=True)
)
print(f'Extracted {len(term_mentions):,} term mentions and {len(term_summary):,} unique terms.')
display(term_summary.head(20))


Extracted 1,727 term mentions and 118 unique terms.


,normalized_term,term,source_columns,network_keys,n_mentions
0,adaptive control,Adaptive control,cognitive_terms,frontoparietal_control,18
1,angular gyrus,Angular gyrus,region_terms,default_mode; language,16
2,anterior insula,Anterior insula,region_terms,cingulo_opercular,14
3,anterior midline,Anterior midline,region_terms,cingulo_opercular,14
4,anterior prefrontal,Anterior prefrontal,region_terms,cingulo_opercular; frontoparietal_control,32
5,anterior temporal lobe,Anterior temporal lobe,region_terms,language,3
6,attention,Attention,network_name,attention,5
7,auditory,Auditory,network_name,auditory,5
8,auditory perception,Auditory perception,cognitive_terms,auditory,5
9,autobiographical memory,Autobiographical memory,cognitive_terms,default_mode,13


In [4]:
from neurovlm.semantic_evaluation import definition_lookup_from_dataframe
from neurovlm.retrieval_resources import (
    _load_cogatlas_dataset,
    _load_cogatlas_task_dataset,
    _load_cogatlas_disorder_dataset,
    _load_kg_mesh_dataset,
)

SOURCE_LOADERS = [
    ('cogatlas', _load_cogatlas_dataset),
    ('cogatlas_task', _load_cogatlas_task_dataset),
    ('cogatlas_disorder', _load_cogatlas_disorder_dataset),
    ('kg_mesh', _load_kg_mesh_dataset),
]

def lookup_from_frame(df, source_name):
    base = definition_lookup_from_dataframe(df)
    rows = []
    for key, definition in base.items():
        if not definition:
            continue
        norm = normalize_network_term(key)
        if norm:
            rows.append({
                'normalized_term': norm,
                'matched_source_term': key,
                'definition': str(definition).strip(),
                'definition_source': source_name,
            })
    return pd.DataFrame(rows).drop_duplicates('normalized_term', keep='first')

source_tables = []
for source_name, loader in SOURCE_LOADERS:
    try:
        df = loader()
        table = lookup_from_frame(df, source_name)
        source_tables.append(table)
        print(f'{source_name}: {len(table):,} definition lookup entries from columns {list(df.columns)}')
    except Exception as exc:
        print(f'WARNING: could not load {source_name}: {exc}')

lookup_df = pd.concat(source_tables, ignore_index=True) if source_tables else pd.DataFrame(columns=['normalized_term', 'matched_source_term', 'definition', 'definition_source'])
lookup_df = lookup_df.drop_duplicates('normalized_term', keep='first')
print('Combined lookup entries:', len(lookup_df))
lookup_df.head()


cogatlas: 906 definition lookup entries from columns ['term', 'definition', 'alias']
cogatlas_task: 843 definition lookup entries from columns ['term', 'definition', 'alias']
cogatlas_disorder: 221 definition lookup entries from columns ['term', 'definition', 'alias']
kg_mesh: 32,993 definition lookup entries from columns ['term', 'definition']
Combined lookup entries: 34789


,normalized_term,matched_source_term,definition,definition_source
0,abductive reasoning,abductive reasoning,The process of adopting an explanatory hypothe...,cogatlas
1,abstract analogy,abstract analogy,High-level analogy that retains general inform...,cogatlas
2,abstract knowledge,abstract knowledge,Knowledge that is general and not tied to a sp...,cogatlas
3,acoustic coding,acoustic coding,A type of short term memory coding that repres...,cogatlas
4,acoustic encoding,acoustic encoding,The processing and encoding of auditory input ...,cogatlas


In [5]:
# Fill definitions by exact normalized-term match against CogAtlas first, then MeSH.
corpus = term_summary.merge(
    lookup_df,
    on='normalized_term',
    how='left',
)
corpus['definition'] = corpus['definition'].fillna('').astype(str).str.strip()
corpus['definition_source'] = corpus['definition_source'].fillna('').astype(str)
corpus['matched_source_term'] = corpus['matched_source_term'].fillna('').astype(str)
corpus['needs_llm_definition'] = corpus['definition'].eq('')
corpus['text'] = [
    f'{term} [SEP] {definition}' if definition else str(term)
    for term, definition in zip(corpus['term'], corpus['definition'])
]

print('Definitions found from CogAtlas/MeSH:', int((~corpus['needs_llm_definition']).sum()))
print('Still missing definitions:', int(corpus['needs_llm_definition'].sum()))
display(corpus.sort_values(['needs_llm_definition', 'term'], ascending=[False, True]).head(30))


Definitions found from CogAtlas/MeSH: 49
Still missing definitions: 69


,normalized_term,term,source_columns,network_keys,n_mentions,matched_source_term,definition,definition_source,needs_llm_definition,text
1,angular gyrus,Angular gyrus,region_terms,default_mode; language,16,,,,True,Angular gyrus
3,anterior midline,Anterior midline,region_terms,cingulo_opercular,14,,,,True,Anterior midline
4,anterior prefrontal,Anterior prefrontal,region_terms,cingulo_opercular; frontoparietal_control,32,,,,True,Anterior prefrontal
5,anterior temporal lobe,Anterior temporal lobe,region_terms,language,3,,,,True,Anterior temporal lobe
7,auditory,Auditory,network_name,auditory,5,,,,True,Auditory
10,autonomic coordination,Autonomic coordination,cognitive_terms,cingulo_opercular,14,,,,True,Autonomic coordination
12,bottom up reorienting,Bottom-up reorienting,cognitive_terms,attention,5,,,,True,Bottom-up reorienting
14,cingulo opercular,Cingulo-Opercular,network_name,cingulo_opercular,14,,,,True,Cingulo-Opercular
17,default mode,Default Mode,network_name,default_mode,13,,,,True,Default Mode
18,dorsal anterior cingulate cortex,Dorsal anterior cingulate cortex (dACC),region_terms,cingulo_opercular,14,,,,True,Dorsal anterior cingulate cortex (dACC)


In [6]:
# Optional Ollama fill for missing definitions.
# Start Ollama locally first, e.g. in a terminal:
#   ollama serve
#   ollama pull qwen2.5:14b-instruct
# If memory is tight, try qwen2.5:7b-instruct or llama3.1:8b.

USE_OLLAMA_FOR_MISSING = True
OLLAMA_MODEL = os.environ.get('NETWORK_TERM_OLLAMA_MODEL', 'qwen2.5:14b-instruct')
MAX_OLLAMA_TERMS = None  # set to an integer for a test batch

SYSTEM_PROMPT = (
    'You write concise neuroscience glossary definitions for retrieval evaluation. '
    'Return only JSON with one key, "definition". The definition must be one plain sentence, '
    '12-35 words, no citations, no markdown.'
)

def parse_ollama_definition(raw):
    try:
        payload = json.loads(raw)
        if isinstance(payload, dict):
            text = payload.get('definition', '')
        else:
            text = ''
    except json.JSONDecodeError:
        text = raw
    text = re.sub(r'\s+', ' ', str(text)).strip().strip('"')
    if text.lower().startswith('definition:'):
        text = text.split(':', 1)[1].strip()
    return text

def ollama_define_term(term):
    import ollama

    user_prompt = (
        'Define this neuroscience/cognitive/anatomical term for semantic retrieval.\n'
        f'Term: {term}\n\n'
        'Return JSON exactly like: {"definition": "..."}'
    )
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        format='json',
    )
    return parse_ollama_definition(response['message']['content'])

missing_idx = corpus.index[corpus['needs_llm_definition']].tolist()
if MAX_OLLAMA_TERMS is not None:
    missing_idx = missing_idx[:MAX_OLLAMA_TERMS]

if not missing_idx:
    print('No missing definitions.')
elif not USE_OLLAMA_FOR_MISSING:
    print(f'{len(missing_idx)} terms are missing definitions. Set USE_OLLAMA_FOR_MISSING=True to fill them with Ollama.')
else:
    print(f'Filling {len(missing_idx)} definitions with Ollama model {OLLAMA_MODEL}...')
    try:
        import ollama  # noqa: F401
    except ImportError as exc:
        raise ImportError('Install the Ollama Python package first: pip install ollama') from exc

    for idx in tqdm(missing_idx):
        term = corpus.at[idx, 'term']
        try:
            definition = ollama_define_term(term)
            if not definition:
                raise ValueError('empty definition returned')
            corpus.at[idx, 'definition'] = definition
            corpus.at[idx, 'definition_source'] = f'ollama:{OLLAMA_MODEL}'
            corpus.at[idx, 'matched_source_term'] = term
            corpus.at[idx, 'needs_llm_definition'] = False
            corpus.at[idx, 'text'] = f'{term} [SEP] {definition}'
            time.sleep(0.2)
        except Exception as exc:
            print(f'WARNING: Ollama definition failed for {term!r}: {exc}')

print('Remaining missing:', int(corpus['definition'].eq('').sum()))
display(corpus[corpus['definition'].eq('')].head(30))


Filling 69 definitions with Ollama model qwen2.5:14b-instruct...


  0%|          | 0/69 [00:00<?, ?it/s]

Remaining missing: 0


,normalized_term,term,source_columns,network_keys,n_mentions,matched_source_term,definition,definition_source,needs_llm_definition,text


In [7]:
# Save the reusable corpus.
column_order = [
    'term',
    'normalized_term',
    'definition',
    'definition_source',
    'matched_source_term',
    'source_columns',
    'network_keys',
    'n_mentions',
    'needs_llm_definition',
    'text',
]
corpus = corpus[column_order].sort_values('term', key=lambda s: s.str.lower()).reset_index(drop=True)
corpus.to_csv(OUTPUT_CSV, index=False)
corpus.to_json(OUTPUT_JSON, orient='records', indent=2)
missing = corpus[corpus['definition'].fillna('').eq('')]
missing.to_csv(MISSING_CSV, index=False)
print(f'Saved {len(corpus):,} terms to {OUTPUT_CSV}')
print(f'Saved JSON to {OUTPUT_JSON}')
print(f'Missing definitions: {len(missing):,}; saved to {MISSING_CSV}')
display(corpus.head(20))


Saved 118 terms to /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_terms_with_definitions.csv
Saved JSON to /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_terms_with_definitions.json
Missing definitions: 0; saved to /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_terms_missing_definitions.csv


,term,normalized_term,definition,definition_source,matched_source_term,source_columns,network_keys,n_mentions,needs_llm_definition,text
0,Adaptive control,adaptive control,Modifying the control law used by a controller...,cogatlas,adaptive control,cognitive_terms,frontoparietal_control,18,False,Adaptive control [SEP] Modifying the control l...
1,Angular gyrus,angular gyrus,Angular gyrus is a brain region involved in in...,ollama:qwen2.5:14b-instruct,Angular gyrus,region_terms,default_mode; language,16,False,Angular gyrus [SEP] Angular gyrus is a brain r...
2,Anterior insula,anterior insula,The anterior insula is a region within the Ins...,kg_mesh,anterior insula,region_terms,cingulo_opercular,14,False,Anterior insula [SEP] The anterior insula is a...
3,Anterior midline,anterior midline,Anterior midline refers to the central axis in...,ollama:qwen2.5:14b-instruct,Anterior midline,region_terms,cingulo_opercular,14,False,Anterior midline [SEP] Anterior midline refers...
4,Anterior prefrontal,anterior prefrontal,The anterior prefrontal cortex is involved in ...,ollama:qwen2.5:14b-instruct,Anterior prefrontal,region_terms,cingulo_opercular; frontoparietal_control,32,False,Anterior prefrontal [SEP] The anterior prefron...
5,Anterior temporal lobe,anterior temporal lobe,Involved in semantic memory and understanding ...,ollama:qwen2.5:14b-instruct,Anterior temporal lobe,region_terms,language,3,False,Anterior temporal lobe [SEP] Involved in seman...
6,Attention,attention,Used to describe any number of cognitive proce...,cogatlas,attention,network_name,attention,5,False,Attention [SEP] Used to describe any number of...
7,Auditory,auditory,Auditory relates to the perception of sound an...,ollama:qwen2.5:14b-instruct,Auditory,network_name,auditory,5,False,Auditory [SEP] Auditory relates to the percept...
8,Auditory perception,auditory perception,"The ability to identify, interpret, and attach...",cogatlas,auditory perception,cognitive_terms,auditory,5,False,Auditory perception [SEP] The ability to ident...
9,Autobiographical memory,autobiographical memory,Autobiographical memory refers to memory for o...,cogatlas,autobiographical memory,cognitive_terms,default_mode,13,False,Autobiographical memory [SEP] Autobiographical...


In [8]:
# Quick sanity checks for the evaluator.
assert OUTPUT_CSV.exists()
assert corpus['normalized_term'].is_unique
assert {'network_name', 'cognitive_terms', 'region_terms'}.issubset(set('; '.join(corpus['source_columns']).split('; ')))
print(corpus['definition_source'].replace('', 'missing').value_counts().to_string())


definition_source
ollama:qwen2.5:14b-instruct    69
cogatlas                       29
kg_mesh                        20
